# Page 17 — "When Reasoning Breaks", measured

Page 17 names three failure modes: unaided self-correction, verifier reliability, and
overthinking. Until now it argued all three from published findings plus an authored Markov
chain with five invented constants. This notebook replaces what it can with the deck's own
runs, and is explicit about the one it cannot.

**It generates nothing.** Every number below is a re-aggregation of artifacts pages 5 to 9
already published. No GPU, no weights, no network unless you point it at the published zips.

What it emits that no other page's `derived.json` already carries:

- the **rung ledger**: answers gained *and lost* at every budget doubling, which page 6's
  climbing curve nets out
- the **affected items**: those answered correctly when cut off early and forced to commit, and
  incorrectly when allowed to run to their own stopping point
- **three controls** on that observation, all of which cut it down, and which the page is
  required to print beside it
- the **specimen**, `test/intermediate_algebra/190.json`, whose traces live only in page 6's
  published zip and not in its `derived.json`

Writes `out/page_17/derived.json` and `manifest.json`.

Rationale lives in [`docs/page_17_refactor.md`](../docs/page_17_refactor.md); the evidence,
including what failed, in [`docs/page_17_findings.md`](../docs/page_17_findings.md).

## How to run this

| Mode | What runs | Needs | Time |
|---|---|---|---|
| `REPLAY` | everything. This notebook has no other mode | pages 5-9's `out/`, no GPU | ~1 min |

**Inputs.** Unpack the published zips for pages 5, 6, 7, 8 and 9 so their `out/` directories
land beside this file, or run those notebooks. Pages 5 and 6 are read as raw traces because
this page re-aggregates them; pages 7, 8 and 9 are read as `derived.json` only, because what
this page needs from them they already publish.

**No `harness` import, deliberately.** `harness.py` is model-loading and generation plumbing,
none of which runs here. Recording its fingerprint would name machinery this run never touched.
What is recorded instead is each **upstream** run's own fingerprint, which is the plumbing that
actually made these numbers, plus this notebook's sha256.

**Gates before figures.** Gate A asserts the runs are one population. Gate B recomputes page 6's
fifteen published accuracy cells and requires them to match exactly. Nothing below gate B may be
trusted before it passes.

In [ ]:
# ---------------------------------------------------------------------------
# The knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "REPLAY"           # the only mode: this notebook generates nothing

PRIMARY = "Qwen/Qwen2.5-3B-Instruct"        # Act 2's shared instrument
FAMILY = ["Qwen/Qwen2.5-1.5B-Instruct",     # page 6's roster, smallest first
          "Qwen/Qwen2.5-3B-Instruct",
          "Qwen/Qwen2.5-7B-Instruct"]

BUDGETS = [128, 256, 512, 1024, 2048]       # page 6's ladder
ITEM_SET = "9ab2b579204221a0c701e6c54ec2c041a63500f9d633ba208aebef8606c43155"
SEED = 123

# The specimen. Chosen before the controls were run, on one criterion the page states out
# loud: its error must be checkable by a reader in seconds. It assumes x = y = z on a problem
# whose minimum is not at the symmetric point. It is also the only item that regresses on all
# three checkpoints.
SPECIMEN = "test/intermediate_algebra/190.json"

# Where the chain goes wrong, and where it commits. Declared here rather than found in
# the page, because "this is the false step" is a claim and claims belong in the notebook
# that publishes them, next to the trace a reader can check them against. Both are
# asserted below, so a trace that ever changes fails the run rather than quietly
# rendering an unlit specimen.
SPECIMEN_ERROR_MARK = r"Assume \(x = y = z\)"

# Control 3's fixed-cut baseline: the "just stop early" rule a reader will propose the moment
# they see the specimen. 512 is the rung where page 6's curve is steepest, so it is the most
# favourable single cut to test the rule at.
FIXED_CUT = 512

# REPLAY inputs when the sibling runs are not already unpacked beside this notebook.
# None means "already in out/", which is how the published zips unpack.
PAGE_05_ZIP_URL = None
PAGE_06_ZIP_URL = None

In [ ]:
import hashlib
import json
import re
import subprocess
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from statistics import median

OUT_DIR = Path("out/page_17")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

UP = {n: Path(f"out/page_{n:02d}") for n in (5, 6, 7, 8, 9)}

NOTEBOOK = "page_17_when_reasoning_breaks.ipynb"


def file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                              text=True, check=True).stdout.strip()
    except Exception:
        return None


def read_traces(path):
    """Page 11's reader: last write per key wins, so a resumed run is one record. The key
    carries every axis these two files use, including page 6's `waits`, or its extension
    arm would collapse to one record per problem and the printed count would understate
    the run."""
    latest = {}
    for line in Path(path).open(encoding="utf-8"):
        if not line.strip():
            continue
        r = json.loads(line)
        latest[(r["run_id"], r["model_id"], r.get("condition"), r.get("budget"),
                r.get("waits"), r["problem_id"], r.get("sample_index", 0))] = r
    return list(latest.values())


def pct(x, digits=1):
    return f"{100 * x:.{digits}f}%"


FAIL = []


def check(name, ok, detail=""):
    """A gate line. Collected rather than raised, so one run prints every failure instead of
    stopping at the first."""
    line = f"  {'ok  ' if ok else 'FAIL'}  {name}{'  ' + detail if detail else ''}"
    print(line)
    if not ok:
        FAIL.append(line)
    return ok


print(f"run mode {RUN_MODE}   notebook {file_sha256(NOTEBOOK)[:12] if Path(NOTEBOOK).exists() else 'unsaved'}")

## 1. The runs this page re-reads

Five runs, one item set. Pages 5 and 6 are loaded as raw traces because their per-item records
are what this page re-aggregates. Pages 7, 8 and 9 are loaded as `derived.json`, because what
page 17 needs from them is already published there and re-deriving it would only risk drift.

In [ ]:
SRC = {}
for n in (5, 6, 7, 8, 9):
    d = UP[n]
    if not (d / "derived.json").exists():
        print(f"page {n}: {d}/ missing -- unpack its published zip beside this notebook")
        continue
    entry = dict(derived=json.loads((d / "derived.json").read_text(encoding="utf-8")),
                 manifest=json.loads((d / "manifest.json").read_text(encoding="utf-8")))
    if n in (5, 6):
        entry["traces"] = read_traces(d / "traces.jsonl")
        entry["traces_sha256"] = file_sha256(d / "traces.jsonl")
    SRC[n] = entry
    m = entry["manifest"]
    rid = m.get("run_id") or m.get("score_id")
    n_rec = f"{len(entry['traces']):,} records" if "traces" in entry else "derived only"
    print(f"page {n}: {n_rec:<16} run {rid}")

assert set(SRC) == {5, 6, 7, 8, 9}, "every upstream run is required; see the cell above"

P6 = SRC[6]["traces"]
P5 = SRC[5]["traces"]

## 2. Gate A — is this one population?

Page 17 says "the same hundred problems" out loud, so it has to be true. Pages 5, 6, 7 and 9
each declare an item-set hash in their manifest and all four must agree. Page 8 is the exception
and it is recorded rather than asserted: it declares no item-set hash, and its own
`page_7_agreement.same_candidates` is **false**, because page 8 generated its own pool over the
same problems. Same items, different chains. The page may say "the same problems" of page 8 and
may not say "the same chains".

In [ ]:
GATE_A = {"declared": {}, "page_08_same_chains": None}

for n in (5, 6, 7, 9):
    m = SRC[n]["manifest"]
    GATE_A["declared"][str(n)] = m["items"]["item_set_sha256"]

print("declared item-set hashes")
agree = all(v == ITEM_SET for v in GATE_A["declared"].values())
for n, h in GATE_A["declared"].items():
    print(f"  page {n}: {h[:16]}  {'==' if h == ITEM_SET else '!='} expected")

# Pages 5 and 6 are re-aggregated here, so their item sets are checked against each other
# directly rather than only via the hash each one asserts about itself.
ids5 = {r["problem_id"] for r in P5}
ids6 = {r["problem_id"] for r in P6}
GATE_A["n_items_page_05"] = len(ids5)
GATE_A["n_items_page_06"] = len(ids6)
GATE_A["sets_identical"] = ids5 == ids6

GATE_A["page_08_same_chains"] = SRC[8]["derived"]["page_7_agreement"]["same_candidates"]
GATE_A["page_09_chain_source"] = SRC[9]["derived"]["chain_provenance"]["source"]

print()
check("pages 5, 6, 7 and 9 declare the same item set", agree)
check("pages 5 and 6 hold the same 100 problem ids", ids5 == ids6 and len(ids6) == 100,
      f"{len(ids6)} ids")
check("page 8 is the same problems but its own chains, and is recorded as such",
      GATE_A["page_08_same_chains"] is False)
check("page 9 reads page 7's pile", GATE_A["page_09_chain_source"] == "page_07_pile")

GATE_A["passed"] = agree and ids5 == ids6 and len(ids6) == 100

## 3. Gate B — page 6's sweep, recomputed

Fifteen cells: three checkpoints by five budgets. Every one must equal the number page 6
shipped. If this drifts, the deck would show one measurement with two values, and everything
below is a re-aggregation of exactly these records.

In [ ]:
GRID = {}                       # model -> problem -> budget -> record
for r in P6:
    if r["budget"] in BUDGETS:
        GRID.setdefault(r["model_id"], {}).setdefault(r["problem_id"], {})[r["budget"]] = r

# Only items with a complete ladder are usable; a partial row cannot be read for a regression.
for m in list(GRID):
    GRID[m] = {p: v for p, v in GRID[m].items() if len(v) == len(BUDGETS)}

GATE_B = {"cells": [], "max_drift": 0.0}
published = {c["model_id"]: {p["budget"]: p["accuracy"] for p in c["points"]}
             for c in SRC[6]["derived"]["figure_2"]["curves"]}

for m in FAMILY:
    for b in BUDGETS:
        got = sum(1 for v in GRID[m].values() if v[b]["correct"]) / len(GRID[m])
        want = published[m][b]
        drift = abs(got - want)
        GATE_B["max_drift"] = max(GATE_B["max_drift"], drift)
        GATE_B["cells"].append(dict(model_id=m, budget=b, recomputed=got, published=want,
                                    drift=drift))

print(f"  {'checkpoint':<26}" + "".join(f"{b:>8}" for b in BUDGETS))
for m in FAMILY:
    row = [c for c in GATE_B["cells"] if c["model_id"] == m]
    print(f"  {m.split('/')[-1]:<26}" + "".join(f"{c['recomputed']:>8.2f}" for c in row))

print()
GATE_B["passed"] = GATE_B["max_drift"] == 0.0
check("all 15 cells reproduce page 6 exactly", GATE_B["passed"],
      f"max drift {GATE_B['max_drift']}")
check("every checkpoint has a complete 100-item ladder",
      all(len(GRID[m]) == 100 for m in FAMILY),
      ", ".join(f"{m.split('/')[-1]}={len(GRID[m])}" for m in FAMILY))

## 4. The rung ledger

Page 6's curve climbs 7, 17, 35, 44, 43 at the primary. That is a net. Here are the two numbers
it is the difference of: answers gained at each doubling, and answers lost.

The last rung is the one to read carefully, and section 5 says why it is nearly empty by
construction rather than by finding.

In [ ]:
def ledger(model_id):
    g = GRID[model_id]
    rows = []
    for i in range(len(BUDGETS) - 1):
        lo, hi = BUDGETS[i], BUDGETS[i + 1]
        gained = sum(1 for v in g.values() if not v[lo]["correct"] and v[hi]["correct"])
        lost = sum(1 for v in g.values() if v[lo]["correct"] and not v[hi]["correct"])
        rows.append(dict(from_budget=lo, to_budget=hi, gained=gained, lost=lost,
                         net=gained - lost))
    return rows


LEDGER = {m: ledger(m) for m in FAMILY}

# Why the top rung is nearly empty: under greedy decoding a chain that has already stopped
# itself is not regenerated at a larger budget, it is the same generation. So above the
# model's own stopping point the axis has no room left. This is the single most important
# caveat on the page and it is measured, not asserted.
SELF_TERMINATED = {}
for m in FAMILY:
    g = GRID[m]
    SELF_TERMINATED[m] = dict(
        share_at_max=sum(1 for v in g.values() if not v[BUDGETS[-1]]["truncated"]) / len(g),
        mean_own_tokens_at_max=sum(v[BUDGETS[-1]]["own_tokens"] for v in g.values()) / len(g),
        truncation_by_budget={b: sum(1 for v in g.values() if v[b]["truncated"]) / len(g)
                              for b in BUDGETS})

print(f"  {'rung':<16}" + "".join(f"{m.split('-')[-2]:>22}" for m in FAMILY))
for i in range(len(BUDGETS) - 1):
    cells = "".join(f"{LEDGER[m][i]['gained']:>8} gained{LEDGER[m][i]['lost']:>6} lost"
                    for m in FAMILY)
    print(f"  {BUDGETS[i]:>5} to {BUDGETS[i+1]:<6}" + cells)

print()
for m in FAMILY:
    s = SELF_TERMINATED[m]
    print(f"  {m.split('/')[-1]:<26} {pct(s['share_at_max'])} self-terminated by "
          f"{BUDGETS[-1]}, mean own length {s['mean_own_tokens_at_max']:.0f} tokens")

## 5. The affected items

Two counts, kept deliberately distinct because they answer different questions and were
conflated once already during this investigation:

- a **regression** is any right-to-wrong step anywhere in the ladder
- a **rescued item** is one whose answer at the largest budget is wrong but which was right at
  some smaller cut

The second is the one page 17 talks about. The 7B has more regressions than rescued items
because two of its dips recover before the top of the ladder.

In [ ]:
def affected(model_id):
    g = GRID[model_id]
    regressions, rescued = [], []
    for p, v in g.items():
        steps = [(BUDGETS[i], BUDGETS[i + 1]) for i in range(len(BUDGETS) - 1)
                 if v[BUDGETS[i]]["correct"] and not v[BUDGETS[i + 1]]["correct"]]
        if steps:
            regressions.append(dict(problem_id=p, level=v[BUDGETS[0]]["level"],
                                    lost_at=[hi for _, hi in steps]))
        if not v[BUDGETS[-1]]["correct"]:
            cuts = [b for b in BUDGETS[:-1] if v[b]["correct"]]
            if cuts:
                rescued.append(dict(
                    problem_id=p, level=v[BUDGETS[0]]["level"],
                    subject=v[BUDGETS[0]]["subject"],
                    right_when_cut_at=cuts,
                    natural_length=v[BUDGETS[-1]]["own_tokens"],
                    exceeded_cap=bool(v[BUDGETS[-1]]["truncated"])))
    return regressions, rescued


AFFECTED = {}
for m in FAMILY:
    regs, resc = affected(m)
    AFFECTED[m] = dict(n_regressions=len(regs), n_rescued=len(resc),
                       regressions=regs, rescued=resc)
    print(f"  {m.split('/')[-1]:<26} {len(regs):>2} regressions   {len(resc):>2} rescued")

# The one aggregate claim the page makes from this: the count falls as the model improves.
SHRINKS = [AFFECTED[m]["n_rescued"] for m in FAMILY]
print()
check("the rescued count falls monotonically as the checkpoint grows",
      SHRINKS == sorted(SHRINKS, reverse=True), " -> ".join(map(str, SHRINKS)))

# The level split, run because page 17's authored figure claimed easy problems break sooner.
# It does not, and the page must not claim it. Kept in the artifact as a stated null.
pooled = [(r["level"], hi) for m in FAMILY for r in AFFECTED[m]["regressions"]
          for hi in r["lost_at"]]
LEVEL_NULL = {str(lv): dict(n=sum(1 for l, _ in pooled if l == lv),
                            median_lost_at=median([b for l, b in pooled if l == lv]))
              for lv in (4, 5)}
LEVEL_NULL["pooled_events"] = len(pooled)
print(f"  level split of {len(pooled)} pooled regression events: "
      + ", ".join(f"L{k} n={v['n']} median {v['median_lost_at']:.0f}"
                  for k, v in LEVEL_NULL.items() if k in ("4", "5")))

## 6. Three controls, all of which cut the observation down

The rescued items look like a finding. Each control below was run to break it, and each one
partly succeeded. **The page is required to print all three beside the specimen**; without them
the reader is handed a strategy that does not exist.

1. **Was the truncated reasoning correct?** No. Every rescue was cut mid-stream with no
   concluding step, so the answer came from the forced commit, not from finished work.
2. **Did the model already know the answer?** Only sometimes, so this is not latent knowledge
   surfacing. Page 5's zero-token blurt arm is the control.
3. **Is stopping early a strategy?** No. A fixed early cut is worse than letting the model
   finish, and cut points buy no more coverage per token than plain resampling does.

In [ ]:
BOX = re.compile(r"\\boxed\{")


def work_of(rec):
    """The chain as written, before the harness appended its forced commit."""
    return rec["trace"].rsplit("Final answer", 1)[0]


# --- control 1: did the cut chain conclude anything? ------------------------------
cases = [(m, r["problem_id"], b)
         for m in FAMILY for r in AFFECTED[m]["rescued"] for b in r["right_when_cut_at"]]
concluded = sum(1 for m, p, b in cases if BOX.search(work_of(GRID[m][p][b])))

# The base rates that make that number readable: a finished chain nearly always boxes an
# answer in its work, a cut one nearly never does.
nat = [v[BUDGETS[-1]] for m in FAMILY for v in GRID[m].values()
       if not v[BUDGETS[-1]]["truncated"]]
cut = [v[b] for m in FAMILY for v in GRID[m].values() for b in BUDGETS[:-1] if v[b]["truncated"]]
CONTROL_1 = dict(
    n_cases=len(cases), n_with_a_concluding_step=concluded,
    boxed_share_when_finished=sum(1 for r in nat if BOX.search(work_of(r))) / len(nat),
    boxed_share_when_cut=sum(1 for r in cut if BOX.search(work_of(r))) / len(cut),
    n_finished=len(nat), n_cut=len(cut))
print(f"1. {concluded} of {len(cases)} rescues had concluded anything when cut "
      f"(a finished chain boxes an answer {pct(CONTROL_1['boxed_share_when_finished'])} "
      f"of the time, a cut one {pct(CONTROL_1['boxed_share_when_cut'])})")

# --- control 2: page 5's blurt arm on the same items ------------------------------
p5 = {(r["model_id"], r["problem_id"], r["condition"]): r["correct"] for r in P5}
items = {(m, r["problem_id"]) for m in FAMILY for r in AFFECTED[m]["rescued"]}
CONTROL_2 = dict(
    n_items=len(items),
    forced_blurt_right=sum(1 for m, p in items if p5.get((m, p, "forced_blurt"))),
    zero_shot_cot_right=sum(1 for m, p in items if p5.get((m, p, "zero_shot_cot"))))
print(f"2. of {CONTROL_2['n_items']} affected items, page 5's zero-token blurt gets "
      f"{CONTROL_2['forced_blurt_right']} and its zero-shot CoT gets "
      f"{CONTROL_2['zero_shot_cot_right']}")

# --- control 3: is there a rule? --------------------------------------------------
g = GRID[PRIMARY]
natural = sum(1 for v in g.values() if v[BUDGETS[-1]]["correct"]) / len(g)
fixed = sum(1 for v in g.values() if v[FIXED_CUT]["correct"]) / len(g)
oracle = sum(1 for v in g.values() if any(v[b]["correct"] for b in BUDGETS)) / len(g)

# What the oracle costs: one generation, whose prefixes are free under greedy, plus a forced
# commit at each earlier cut. Compared against page 7's measured coverage-per-token curve on
# the same items, by linear interpolation between its bracketing points. The curve is concave,
# so its chord understates it, which makes this comparison generous to the cut points.
allowance = SRC[6]["manifest"]["forcing"]["answer_allowance"]
cost = SELF_TERMINATED[PRIMARY]["mean_own_tokens_at_max"] + allowance * (len(BUDGETS) - 1)
pile = next(p for p in SRC[7]["derived"]["figure_2"]["pile"] if p["temperature"] > 0)
pts = sorted(pile["points"], key=lambda p: p["mean_tokens"])
lo = max((p for p in pts if p["mean_tokens"] <= cost), key=lambda p: p["mean_tokens"])
hi = min((p for p in pts if p["mean_tokens"] > cost), key=lambda p: p["mean_tokens"])
frac = (cost - lo["mean_tokens"]) / (hi["mean_tokens"] - lo["mean_tokens"])
sampled = lo["coverage"] + frac * (hi["coverage"] - lo["coverage"])

CONTROL_3 = dict(
    model_id=PRIMARY, natural=natural, fixed_cut=FIXED_CUT, fixed_cut_accuracy=fixed,
    oracle_over_cuts=oracle, oracle_token_cost=cost, answer_allowance=allowance,
    sampling_coverage_at_that_cost=sampled, sampling_temperature=pile["temperature"],
    sampling_bracket=[dict(n=lo["n"], coverage=lo["coverage"], mean_tokens=lo["mean_tokens"]),
                      dict(n=hi["n"], coverage=hi["coverage"], mean_tokens=hi["mean_tokens"])],
    interpolation="linear between the bracketing points of page 7's pile; the curve is "
                  "concave so its chord understates it, which favours the cut points")
print(f"3. at {PRIMARY.split('/')[-1]}: natural {pct(natural, 0)}, "
      f"a fixed cut at {FIXED_CUT} {pct(fixed, 0)}, best cut point {pct(oracle, 0)} "
      f"for ~{cost:.0f} tokens; sampling reaches {pct(sampled, 0)} at that cost")

check("a fixed early cut is worse than letting the model finish", fixed < natural)
check("cut points buy no more coverage per token than resampling", oracle <= sampled + 0.005)

## 7. The specimen

`test/intermediate_algebra/190.json`. Minimise $(x+y)(y+z)$ given $xyz(x+y+z)=1$; the answer
is 2. The chain assumes $x=y=z$, computes $4/\sqrt{3}\approx 2.309$, and boxes it. AM-GM on
$(x+y)(y+z) = y(x+y+z) + xz \ge 2\sqrt{xyz(x+y+z)}$ gives exactly 2.

The page ships every budget's trace so a reader can find the assumption themselves. This is the
only part of page 17 that cannot be built from `derived.json` files alone, and it is the reason
this notebook exists.

In [ ]:
assert SPECIMEN in GRID[PRIMARY], f"{SPECIMEN} is not in the primary's complete ladder"

# The stem the harness appends when it cuts a chain off and makes it answer. Read from
# page 6's own manifest rather than retyped, so the page cannot light a marker that run
# never used.
FORCE_STEM = SRC[6]["manifest"]["forcing"]["stem"]

v = GRID[PRIMARY][SPECIMEN]
SPECIMEN_ROW = dict(
    problem_id=SPECIMEN, model_id=PRIMARY,
    level=v[BUDGETS[0]]["level"], subject=v[BUDGETS[0]]["subject"],
    question=v[BUDGETS[0]]["question"], ground_truth=v[BUDGETS[0]]["ground_truth"],
    natural_length=v[BUDGETS[-1]]["own_tokens"],
    by_budget=[dict(budget=b, trace=v[b]["trace"], own_tokens=v[b]["own_tokens"],
                    cut_index=v[b]["cut_index"], answer_start=v[b]["answer_start"],
                    truncated=v[b]["truncated"], forced=v[b]["forced"],
                    extracted_answer=v[b]["extracted_answer"], correct=v[b]["correct"],
                    # Where the harness stopped the model and made it commit. NOT
                    # `answer_start`, which is the character the cut landed on: the
                    # commit is the appended stem, and a page that lit `answer_start`
                    # would highlight several hundred characters of the model's own
                    # working and call them the forced answer.
                    commit=(v[b]["trace"].rfind(FORCE_STEM.strip())
                            if v[b]["truncated"] else None))
               for b in BUDGETS])
for _row in SPECIMEN_ROW["by_budget"]:
    if _row["truncated"] and _row["commit"] < 0:
        raise SystemExit(f"the specimen at budget {_row['budget']} was forced but carries "
                         f"no {FORCE_STEM.strip()!r}")
# It regresses on every checkpoint, which is as close to reproducible as one item gets.
SPECIMEN_ROW["also_regresses_on"] = [
    m for m in FAMILY if m != PRIMARY
    and any(r["problem_id"] == SPECIMEN for r in AFFECTED[m]["regressions"])]

# The two spans the page lights, as character offsets into the chain the model wrote
# for itself. The first is the unjustified step; the second is what it committed to.
# Between them is 300-odd tokens of arithmetic that is all correct and all downstream
# of the mistake, which is the thing the reader should see.
natural = v[BUDGETS[-1]]["trace"]
err = natural.find(SPECIMEN_ERROR_MARK)
box = natural.rfind("\\boxed{")
if err < 0:
    raise SystemExit(f"the specimen's declared error mark is not in its trace: "
                     f"{SPECIMEN_ERROR_MARK!r}")
if box < 0 or box < err:
    raise SystemExit("the specimen concludes before it goes wrong, or never boxes")
SPECIMEN_ROW["zones"] = {
    "assumption": [err, err + len(SPECIMEN_ERROR_MARK)],
    "conclusion": [box, len(natural.rstrip())],
    "mark": SPECIMEN_ERROR_MARK,
    "why": "an unjustified symmetry assumption: the constraint is symmetric, the "
           "objective is not, so the minimum is not at x = y = z",
}
print(f"  error at char {err}, conclusion at char {box}, of {len(natural)}")

for row in SPECIMEN_ROW["by_budget"]:
    print(f"  {row['budget']:>5}  {'right' if row['correct'] else 'wrong':<6} "
          f"{'cut and forced' if row['truncated'] else 'ran to its own end':<20} "
          f"{row['own_tokens']:>5} tokens   answer {str(row['extracted_answer'])[:22]}")

right_when_cut = [r["budget"] for r in SPECIMEN_ROW["by_budget"] if r["correct"]]
check("the specimen is right at more than one cut and wrong at its own ending",
      len(right_when_cut) > 1 and not SPECIMEN_ROW["by_budget"][-1]["correct"],
      f"right at {right_when_cut}")
check("the specimen's own ending is not a cap artifact",
      not v[BUDGETS[-1]]["truncated"], f"{SPECIMEN_ROW['natural_length']} tokens")

## 8. Artifacts

The claims block is the contract. A `false` here is not a bug, it is a finding the page must
carry, and two of them are.

In [ ]:
claims = {
    # Gates.
    "one_population": GATE_A["passed"],
    "page_6_reproduced_exactly": GATE_B["passed"],
    # What the page may say.
    # Scoped to the primary on purpose: the 1.5B's top rung is 0 gained and 0 lost, so the
    # unscoped version of this sentence would be false and the page must not write it.
    "every_rung_loses_answers_on_the_primary": all(r["lost"] > 0 for r in LEDGER[PRIMARY]),
    "top_rung_gains_nothing_on_the_primary": LEDGER[PRIMARY][-1]["gained"] == 0,
    "rescued_count_falls_with_capability":
        [AFFECTED[m]["n_rescued"] for m in FAMILY] ==
        sorted([AFFECTED[m]["n_rescued"] for m in FAMILY], reverse=True),
    "specimen_complete": len(SPECIMEN_ROW["by_budget"]) == len(BUDGETS),
    "specimen_regresses_on_every_checkpoint":
        len(SPECIMEN_ROW["also_regresses_on"]) == len(FAMILY) - 1,
    # What the controls forbid. Both false, deliberately, and the page prints both.
    "rescues_came_from_finished_reasoning": CONTROL_1["n_with_a_concluding_step"] > 0,
    "stopping_early_is_a_strategy":
        CONTROL_3["fixed_cut_accuracy"] > CONTROL_3["natural"],
    "cut_points_beat_resampling_per_token":
        CONTROL_3["oracle_over_cuts"] > CONTROL_3["sampling_coverage_at_that_cost"] + 0.005,
    # The claim page 17's authored figure made, tested and dead.
    "easy_problems_break_sooner_than_hard":
        LEVEL_NULL["4"]["median_lost_at"] < LEVEL_NULL["5"]["median_lost_at"],
}

manifest = {
    "run_id": f"p17-replay-{ITEM_SET[:8]}",
    "notebook": NOTEBOOK,
    "notebook_sha256": file_sha256(NOTEBOOK),
    "notebook_git_sha": git_sha(),
    "page": 17,
    "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "spec": "docs/page_17_refactor.md",
    "findings": "docs/page_17_findings.md",
    "generates": "nothing: every number is a re-aggregation of pages 5 to 9",
    # No harness fingerprint of our own, because no harness plumbing ran. What made these
    # numbers is each upstream run's own, recorded here.
    "sources": {str(n): dict(
        run_id=(SRC[n]["manifest"].get("run_id") or SRC[n]["manifest"].get("score_id")),
        notebook=SRC[n]["manifest"].get("notebook"),
        harness=SRC[n]["manifest"].get("harness"),
        read_as="traces" if n in (5, 6) else "derived",
        traces_sha256=SRC[n].get("traces_sha256")) for n in sorted(SRC)},
    "items": {"dataset": "math500_l45", "levels": [4, 5], "n": 100,
              "item_set_sha256": ITEM_SET, "seed": SEED},
    "models": {"primary": PRIMARY, "family": FAMILY},
    "budgets": BUDGETS,
    "decoding": SRC[6]["manifest"]["decoding"],
    "gates": {"A_one_population": GATE_A, "B_page_6_reproduced": GATE_B},
}

derived = {
    "manifest": manifest,
    "claims": claims,
    "procedure": (
        "Re-aggregate page 6's budget sweep per item rather than per population: for each "
        "budget doubling count answers gained and answers lost, and flag every item answered "
        "correctly at a smaller cut and incorrectly at the largest. Three controls test "
        "whether that is a strategy: whether the cut chain had concluded anything, whether "
        "page 5's zero-token blurt already knew the answer, and whether a fixed cut or the "
        "best cut beats letting the model finish at equal token cost. All three cut it down."),
    "ledger": LEDGER,
    "self_terminated": SELF_TERMINATED,
    "affected": {m: {k: v for k, v in AFFECTED[m].items()} for m in FAMILY},
    "controls": {"concluded_when_cut": CONTROL_1, "blurt_arm": CONTROL_2,
                 "is_it_a_rule": CONTROL_3},
    "level_null": LEVEL_NULL,
    "specimen": SPECIMEN_ROW,
}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"wrote {DERIVED_PATH} ({DERIVED_PATH.stat().st_size / 1024:.0f} kB) "
      f"and {MANIFEST_PATH}\n")
for k, val in claims.items():
    print(f"  {str(val):<6} {k}")
if FAIL:
    print("\nGATE FAILURES:")
    for f in FAIL:
        print(f)
    raise SystemExit("gates failed; nothing downstream may be trusted")
print("\nall gates passed")

## Notes for re-derivation

- **This notebook generates nothing.** Every number is a re-aggregation of pages 5 to 9's
  published artifacts, and gate B asserts that the re-aggregation reproduces page 6's own
  fifteen accuracy cells exactly. To re-derive from scratch: unpack the five published zips so
  their `out/` directories sit beside this file, and run.
- **Two claims in the artifact are deliberately `false`.**
  `rescues_came_from_finished_reasoning` and `stopping_early_is_a_strategy` are the controls
  that cut the observation down, and page 17 prints both beside the specimen. A future run that
  flips either one to `true` is a finding, not a fix.
- **`easy_problems_break_sooner_than_hard` is the claim page 17's authored figure used to
  make.** It is tested here on all three checkpoints pooled and it is dead. It is also
  unreachable on this instrument rather than merely underpowered: under greedy decoding a chain
  that has stopped itself is not regenerated at a larger budget, so the budget axis has no room
  above the model's own stopping point. `docs/page_17_findings.md` section 4 has the estimate
  for the experiment that could ask the question, and the recommendation against running it.
- **The specimen is chosen for legibility, not for effect size.** One item cannot carry a
  claim about a population, and the page's aggregate claim is only that the affected count
  falls as the checkpoint grows.